In [4]:
import os
import requests
import pandas as pd

# =========================
# CONFIG
# =========================
MERPIS_API_TOKEN = "Bearer 13|0c2Tk8HsCjzLf2PrLZlrrsHg2GGZkgTbDBetcoTn34b2a8d2"

url = "https://merpis.ptmerahputih.com/api/downtime/3?year=&month=&customer=&tugboat=&barge="

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {MERPIS_API_TOKEN}"
}

BATCH_SIZE = 500

# =========================
# VALIDASI TOKEN
# =========================
if not MERPIS_API_TOKEN:
    raise Exception("MERPIS_API_TOKEN belum tersedia")

# =========================
# REQUEST API
# =========================
response = requests.get(
    url,
    headers=headers,
    timeout=300
)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print(response.text)
    raise Exception("Gagal mengambil data API")

# =========================
# PARSE JSON
# =========================
json_data = response.json()

data = json_data["data"]["data"]

print("Total Raw Data:", len(data))

# =========================
# PROCESS DATA
# =========================
all_rows = []

for start in range(0, len(data), BATCH_SIZE):

    end = start + BATCH_SIZE

    batch = data[start:end]

    print(f"\nProcessing Batch {start} - {min(end, len(data))}")

    batch_rows = []

    for row in batch:

        projectel = row.get("projectel", {})

        tugboat = projectel.get("tugboatel", {})
        barge = projectel.get("bargeel", {})
        customer = projectel.get("customerel", {})

        batch_rows.append({

            "Year": row.get("year"),
            "Month": row.get("month"),

            "Code": projectel.get("code"),

            "Tugboat": tugboat.get("name"),
            "Barge": barge.get("name"),

            "Customer": customer.get("fullname"),

            "ETA Arrived POD": row.get("etaarrivepod"),
            "Actual Arrived POD": row.get("actualarrivepod"),

            "Downtime": row.get("downtime"),

            "Department": row.get("department"),
            "Category": row.get("category"),

            "Notes": row.get("notes")

        })

    print("Batch Rows:", len(batch_rows))

    all_rows.extend(batch_rows)

# =========================
# DATAFRAME
# =========================
df = pd.DataFrame(all_rows)

# =========================
# CONVERT DOWNTIME TO DAYS
# =========================
df["Downtime_Days"] = (
    pd.to_timedelta(
        df["Downtime"],
        errors="coerce"
    ).dt.total_seconds() / 86400
).round(6)

# =========================
# RESULT
# =========================
print("\n==========================")
print("FINAL RESULT")
print("==========================")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(df.head())

print("\nTotal Final Rows :", len(df))
print("Total Columns    :", len(df.columns))

print("\n==========================")
print("AVAILABLE COLUMNS FROM API")
print("==========================")

if len(data) > 0:
    print(data[0].keys())

Status Code: 200
Total Raw Data: 351

Processing Batch 0 - 351
Batch Rows: 351

FINAL RESULT
   Year Month              Code         Tugboat           Barge  \
0  2026    04  BG.MP31-2026-006       TB. MP 37   BG. MP 330 31   
1  2026    04  BG.MP33-2026-006       TB. MP 29   BG. MP 330 33   
2  2026    04  BG.MP35-2026-006       TB. MP 31   BG. MP 330 35   
3  2026    04  BG.PS01-2026-002  TB. PSIP 160.1  BG. PSIP 250.1   
4  2026    04  BG.JY08-2026-004    TB. SEKAR 71     BG. JAYA 08   

                             Customer      ETA Arrived POD  \
0               MEGA MITRA MARINE, PT  2026-04-14 15:10:00   
1          MINERAL MAJU SEJAHTERA, PT  2026-04-18 23:09:00   
2  BIMA CAKRA PERKASA MINERALINDO, PT  2026-04-16 08:40:00   
3          GRACIA ALAYA SEJAHTERA, PT  2026-04-13 16:34:00   
4               LAJU MARITIM JAYA, PT  2026-04-11 04:00:00   

    Actual Arrived POD  Downtime  Department  Category  Notes  Downtime_Days  
0  2026-04-14 18:00:00  02:00:00           1       5